### PIPLINE FOR RAG

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os
from groq import Groq


In [3]:
def getting_all_pdf(pdf_directory):
    all_document=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"find all the  pdf total{len(pdf_files)}")
    for pdf_file in pdf_files:
        print(f"pdf:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            document=loader.load()
            for doc in document:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]="pdf"
            all_document.extend(document)
            print(f"there is {len(document)}pages")
        except Exception as e:
            print(f"some file has issue{e}"
            )
    print(f"total lengthof the document is {len(all_document)}")
    return all_document
all_pdf=getting_all_pdf("../data")

find all the  pdf total3
pdf:agricuture_2.pdf
there is 31pages
pdf:agricuture_3.pdf
there is 186pages
pdf:agri_culture.pdf
there is 67pages
total lengthof the document is 284


In [4]:
all_pdf

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2014-12-16T11:24:49+08:00', 'author': 'Annie S. Wesley and Merle Faminow', 'keywords': 'food security, agriculture, research and extension, asia and pacific, applied research, agriculture research, boosting food production, agriculture production, agricultural productivity, crop diversification, secondary crops, small farms, postharvest food losses, annie wesley, merle faminow, economics working paper 425, ewp 425', 'moddate': '2014-12-22T16:21:23+08:00', 'subject': 'In view of emerging economic, climatic, and political scenarios in the region, this paper explores the role of applied research for development and extension services through a two-pronged approach of boosting food production and preventing losses.', 'title': 'Background Paper: Research and Development and Extension Services in Agriculture and Food Security (Economics Working Paper No. 425)', 'sourc

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_document(document, chunk_size=1000, chunk_overlap=200):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_doc = text_splitter.split_documents(document)

    print(f"Original docs: {len(document)}, Split docs: {len(split_doc)}")

    return split_doc

In [6]:
split_document(all_pdf)


Original docs: 284, Split docs: 671


[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2014-12-16T11:24:49+08:00', 'author': 'Annie S. Wesley and Merle Faminow', 'keywords': 'food security, agriculture, research and extension, asia and pacific, applied research, agriculture research, boosting food production, agriculture production, agricultural productivity, crop diversification, secondary crops, small farms, postharvest food losses, annie wesley, merle faminow, economics working paper 425, ewp 425', 'moddate': '2014-12-22T16:21:23+08:00', 'subject': 'In view of emerging economic, climatic, and political scenarios in the region, this paper explores the role of applied research for development and extension services through a two-pronged approach of boosting food production and preventing losses.', 'title': 'Background Paper: Research and Development and Extension Services in Agriculture and Food Security (Economics Working Paper No. 425)', 'sourc

### EMBEDING


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Tuple,Any
from  sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name   # ✅ store as model_name, not model
        self.model = None
        self._load_model()             # ✅ must match method name below

    def _load_model(self):             # ✅ renamed to _load_model
        try:
            print(f"loading the model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"error in loading: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"generating the embedding shape: {embedding.shape}")
        return embedding

embedding_manager = EmbeddingManager()
embedding_manager

loading the model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1813.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [9]:
test_texts = ["Hello world", "This is a test sentence"]
embeddings = embedding_manager.generate_embedding(test_texts)
print(f"Shape: {embeddings.shape}")  # Expected: (2, 384)
print(f"Sample vector (first 5 dims): {embeddings[0][:5]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s]

generating the embedding shape: (2, 384)
Shape: (2, 384)
Sample vector (first 5 dims): [-0.03447723  0.03102317  0.00673497  0.02610899 -0.03936205]


In [10]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(  # ✅ self.client
                name=self.collection_name,                            # ✅ comma added
                metadata={"description": "Pdf document embedding for RAG"}
            )
            print(self.collection_name)
            print(self.collection.count())
        except Exception as e:
            print(f"error: {e}")
            raise

    def add_document(self, document: List[Any], embeddings: np.ndarray):  # ✅ self
        if len(document) != len(embeddings):
            raise ValueError("number of documents do not match")

        ids = []
        metadatas = []          # ✅ consistent name
        documents_text = []     # ✅ consistent name
        embeddings_list = []    # ✅ consistent name

        for i, (doc, embedding) in enumerate(zip(document, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            meta = dict(doc.metadata)
            meta['doc_index'] = i
            meta['content_length'] = len(doc.page_content)
            metadatas.append(meta)          # ✅ using meta to avoid shadowing list

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(document)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

pdf_documents
671


In [11]:
chunks = split_document(all_pdf)  # ✅ separated and saved to variable
chunks[0]

Original docs: 284, Split docs: 671


Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2014-12-16T11:24:49+08:00', 'author': 'Annie S. Wesley and Merle Faminow', 'keywords': 'food security, agriculture, research and extension, asia and pacific, applied research, agriculture research, boosting food production, agriculture production, agricultural productivity, crop diversification, secondary crops, small farms, postharvest food losses, annie wesley, merle faminow, economics working paper 425, ewp 425', 'moddate': '2014-12-22T16:21:23+08:00', 'subject': 'In view of emerging economic, climatic, and political scenarios in the region, this paper explores the role of applied research for development and extension services through a two-pronged approach of boosting food production and preventing losses.', 'title': 'Background Paper: Research and Development and Extension Services in Agriculture and Food Security (Economics Working Paper No. 425)', 'source

In [12]:
embeddings = embedding_manager.generate_embedding([doc.page_content for doc in chunks])
vector_store.add_document(chunks, embeddings)

Batches: 100%|██████████| 21/21 [00:24<00:00,  1.16s/it]


generating the embedding shape: (671, 384)
Successfully added 671 documents to vector store
Total documents in collection: 1342


### RETRIVAL PIPLINE


In [13]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [14]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [15]:
results = rag_retriever.retrieve(" how can Weed control", top_k=5)
for r in results:
    print(f"Rank {r['rank']} | Score: {r['similarity_score']:.3f}")
    print(r['content'][:200])
    print("---")

Retrieving documents for query: ' how can Weed control'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.92it/s]

generating the embedding shape: (1, 384)
Retrieved 2 documents (after filtering)
Rank 1 | Score: 0.007
suppressing weeds compared to sole cropping (Girjesh and Patil, 1991). 
4. Preservation of natural balance: Different plants in polyculture 
can help maintain a natural balance. For example, some plan
---
Rank 2 | Score: 0.007
suppressing weeds compared to sole cropping (Girjesh and Patil, 1991). 
4. Preservation of natural balance: Different plants in polyculture 
can help maintain a natural balance. For example, some plan
---


In [31]:
from groq import Groq
from typing import List, Dict, Any
import os
import time
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = Groq(api_key=groq_api_key)
MODEL_NAME = "llama-3.3-70b-versatile"
print("Groq client initialized successfully")

Groq client initialized successfully


In [32]:
def call_groq(prompt: str, model: str = MODEL_NAME, max_tokens: int = 1024, temperature: float = 0.1) -> str:
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content

In [ ]:
def rag_simple(query, retriever, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['text'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    return call_groq(prompt)

# Test
answer = rag_simple("How farmer grow crop effectivly?", rag_retriever)
print(answer)

Retrieving documents for query: 'How farmer grow crop effectivly?'
Top K: 3, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.50it/s]

generating the embedding shape: (1, 384)
Retrieved 3 documents (after filtering)


Through intercropping, which allows for more efficient use of soil resources, optimization of water and energy usage, and adaptation to seasons, enabling farmers to grow crops effectively and increase production.


In [36]:
def rag_advanced(query, retriever, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Fix keys to match your retriever's actual output
    context = "\n\n".join([doc['text'] for doc in results])  # was 'content'
    
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': round(doc['score'], 3),          # was 'similarity_score'
        'preview': doc['text'][:300] + '...'      # was 'content'
    } for doc in results]

    confidence = max([doc['score'] for doc in results])  # was 'similarity_score'
    ...

In [43]:
class AdvancedRAGPipeline:
    def __init__(self, retriever):
        self.retriever = retriever
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Step 1: Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        if not results:
            answer = "No relevant context found."
            sources = []
        else:
            context = "\n\n".join([doc['content'] for doc in results])

            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': round(doc['similarity_score'], 3),
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""

            # Step 2: Optional streaming simulation
            if stream:
                print("Streaming answer:")
                for i in range(0, min(len(prompt), 400), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print("\n")

            # Step 3: Generate answer
            answer = call_groq(prompt)

        # Step 4: Add citations
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations else answer
        )

        # Step 5: Optional summary
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary = call_groq(summary_prompt)

        # Step 6: Store history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [47]:
adv_rag = AdvancedRAGPipeline(rag_retriever)

result = adv_rag.query(
    "PHENOLIC METABOLISM IN PLANTS UNDER HEAVY  METAL STRESS ?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:\n", result['answer'])
print("\nSummary:", result['summary'])
print("\nSources:")
for src in result['sources']:
    print(f"  - {src['source']} | Page: {src['page']} | Score: {src['score']}")
print("\nLast History Entry:", result['history'][-1]['question'])

Retrieving documents for query: 'PHENOLIC METABOLISM IN PLANTS UNDER HEAVY  METAL STRESS ?'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.63it/s]

generating the embedding shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.

Context:
101 | RESEARCH TOPICS IN AGRICULTURE 
 
 
 
 
 
 
 
 
 
 
CHAPTER 8 
 
PHENOLIC METABOLISM IN PLANTS UNDER HEAVY  
METAL STRESS 
 
PhD Student Vesile YALCIN1 
Assoc. Prof. Dr. Hülya TORUN2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
1Duzce University, Graduate Sc

hool of Education, Department of Biology, Duzce, Türkiye. 
yalcinvesile42@gmail.


Final Answer:
 Phenolic metabolism in plants is affected by heavy metal stress, which can lead to various phytotoxic damages, but plants use defense mechanisms such as hormones, phytochelatins, and metal tolerance to respond to the stress.

Citations:
[1] agricuture_3.pdf (page 105)
[2] agricuture_3.pdf (page 105)
[3] agricuture_3.pdf (page 107)

Summary: Heavy metal stress can harm plants by disrupting their phenolic metabolism, leading to various forms of damage. However, plants have developed defense mechanisms, including the use of hormones, phytochelatins, and metal tolerance, to respond to and mitigate the effects of heavy metal stress.

Sources:
  - agricuture_3.pdf | Page: 105 | Score: 0.634
  - agricuture_3.pdf | Page: 105 | Score: 0.634
  - agricuture_3.pdf | Page: 107 | Score: 0.406

Last History Entry: PHENOLIC METABOLISM IN PLANTS UNDER HEAVY  METAL STRESS ?


In [46]:
results = rag_retriever.retrieve("PHENOLIC METABOLISM IN PLANTS", top_k=3)
print(results[0].keys())
print(results[0])

Retrieving documents for query: 'PHENOLIC METABOLISM IN PLANTS'
Top K: 3, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.24it/s]

generating the embedding shape: (1, 384)
Retrieved 3 documents (after filtering)
dict_keys(['id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'])
{'id': 'doc_dad9e4d1_374', 'content': 'Safari vd., 2019; Chen et al., 2020; Lwalaba et  al., 2020; Antony ve Nagella, \n2021; Janczak-Pieniazek vd., 2023). \n \n2. CLASSIFICATION AND STRUCTURE OF PHENOLIC \nCOMPOUNDS \nPhenolic metabolism is one of the highly branched networks that provide \nabout 40% of the organic compounds in plants ( Kováčik and Klejdus,  2008). \nPhenolics are second only to hemicellulose, cellulose and lignin and are the \nfourth major component in plants (Yan et al., 2021; Chen et al., 2023). These \ncompounds are often found in the cell walls and vacuoles of epidermal and \nsubepidermal cells in plants (Kisiriko et al., 2021).  They are organic \ncompounds synthesized during the normal development o f plants and in \nresponse to stressful conditions (Shahidi et al., 2019). The family of PCs is \nsecond